# Classification de sentiment des avis TMDB (endpoint `/sentiment`)

**Perimetre** : Personne B (ML & NLP Engineer) -- voir `CLAUDE.md`.

**Objectif** : entrainer un modele de classification de sentiment
(negatif / neutre / positif) sur les avis TMDB stockes dans la table Gold
`avis`, pour alimenter l'endpoint `/sentiment`
(`api/routers/sentiment.py`, schema `SentimentScore`).

**Seuils a respecter** : F1 macro > 0.70 et accuracy > 0.72
(voir `CLAUDE.md`, section `/sentiment`).

**Point d'attention majeur, a garder en tete tout au long du notebook** :
la table `avis` contient de vrais textes d'avis TMDB, mais leur
rattachement a un `user_id` MovieLens est **synthetique** (tirage
aleatoire parmi les utilisateurs ayant reellement note le film, pour
respecter les contraintes de cle etrangere -- voir
`pipeline/transform_silver.py::clean_avis`). Le label de sentiment utilise
ici est donc **derive de la note** laissee par cet utilisateur MovieLens,
et non de la note que l'auteur reel de l'avis aurait pu donner lui-meme.
C'est une supervision faible (distant supervision), structurellement
bruitee : un desaccord entre le label derive de la note et le sentiment
reel du texte est attendu pour une fraction des exemples, pas seulement
possible. Ce point est repris concretement dans la section
"Exemples mal classes" et dans la section "Limites".

**Structure du notebook** :
1. Chargement des donnees Gold (avis + notation)
2. EDA (couverture, distribution des notes/labels, longueur des avis)
3. Derivation du label et repartition des classes
4. Split train/test stratifie (pas de fuite)
5. Baseline naive (classe majoritaire)
6. Baseline classique (TF-IDF + regression logistique)
7. Fine-tuning DistilBERT -- smoke test (~100 exemples)
8. Fine-tuning DistilBERT -- entrainement complet (charge le modele deja
   entraine par `python -m nlp.training.model`)
9. Comparaison des modeles
10. Matrice de confusion
11. F1 par classe
12. Exemples mal classes commentes
13. Discussion des limites
14. Conclusion


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "nlp").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from nlp.training import features
from nlp.training import model as model_module

pd.set_option("display.max_colwidth", 200)
RANDOM_STATE = 42


## 1. Chargement des donnees Gold

Connexion directe a la base Gold (Supabase Postgres) via `DATABASE_URL`,
memes utilitaires que `nlp/training/model.py` (pas de duplication de la
logique de connexion).


In [ ]:
avis_df, notation_df = model_module.load_gold_avis_data()
print(f"avis (Gold) : {len(avis_df)} lignes")
print(f"notation (Gold) : {len(notation_df)} lignes")


## 2. EDA -- couverture, notes, longueur des avis

On construit d'abord le jeu de donnees labelise (jointure avis x notation
+ derivation du label + longueur du texte), puis on regarde :
- la couverture des avis par rapport a l'ensemble des notations,
- la distribution des notes/labels,
- la distribution de la longueur des avis (caracteres et mots).


In [ ]:
labeled_df = model_module.build_labeled_dataset(avis_df, notation_df)
coverage_pct = 100 * len(labeled_df) / len(notation_df)
print(f"Avis labelises : {len(labeled_df)}")
print(f"Couverture avis / notations totales : {coverage_pct:.3f}%")
print()
print("Distribution des labels derives de la note :")
print(features.label_distribution(labeled_df))
print()
print("Distribution des labels (%) :")
print((features.label_distribution(labeled_df) / len(labeled_df) * 100).round(1))


In [ ]:
print("Statistiques de longueur des avis (caracteres) :")
print(labeled_df["longueur_car"].describe())
print()
print("Statistiques de longueur des avis (mots) :")
print(labeled_df["longueur_mots"].describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(labeled_df["longueur_mots"], bins=40, color="steelblue")
axes[0].set_title("Longueur des avis (mots)")
axes[0].set_xlabel("nb mots")

axes[1].bar(
    features.label_distribution(labeled_df).index,
    features.label_distribution(labeled_df).values,
    color=["indianred", "gray", "seagreen"],
)
axes[1].set_title("Distribution des labels derives de la note")
plt.tight_layout()
plt.show()


**Interprétation (EDA)** : seuls 2 082 avis textuels sont disponibles,
contre 1 000 209 notations dans la table Gold -- une couverture d'a peine
**0,208%**. L'echantillon d'entrainement/evaluation est donc tres
restreint face au volume total de la plateforme, ce qui limite d'emblee
la robustesse statistique de toute metrique calculee plus loin (voir
section Limites). La distribution des labels derives de la note est
desequilibree : `positif` domine largement (~58%), suivi de `neutre`
(~24%) et `negatif` (~18%) -- consequence directe de la distribution des
notes MovieLens elle-meme (notes 4-5 majoritaires). Les avis sont en
moyenne longs (plusieurs centaines de mots), ce qui va directement poser
un probleme de troncature au moment de la tokenisation (section 8).


## 3. Derivation du label et repartition des classes

Regle de derivation (`nlp/training/features.py::derive_label_from_note`) :
- note <= 2 -> `negatif`
- note == 3 -> `neutre`
- note >= 4 -> `positif`

**Rappel critique** : cette note est celle de l'utilisateur MovieLens
auquel l'avis a ete rattache *synthetiquement* (tirage aleatoire parmi les
votants reels du film), pas necessairement celle qu'aurait donnee l'auteur
reel de l'avis TMDB. Le label est donc une approximation par supervision
faible, et non une verite terrain fiable a 100%.


In [ ]:
print(labeled_df[["note", "label"]].groupby("label")["note"].describe())


## 4. Split train/test stratifie (pas de fuite)

Split stratifie sur le label (80/20), verification explicite qu'aucune
paire (user_id, film_id) n'apparait a la fois dans le train et le test.


In [ ]:
train_df, test_df = features.stratified_label_train_test_split(
    labeled_df, test_size=0.2, random_state=RANDOM_STATE
)

train_pairs = set(zip(train_df["user_id"], train_df["film_id"]))
test_pairs = set(zip(test_df["user_id"], test_df["film_id"]))
assert train_pairs.isdisjoint(test_pairs), "Fuite train/test detectee !"

print(f"Train : {len(train_df)}  Test : {len(test_df)}")
print("Pas de fuite train/test (paires user_id/film_id disjointes).")
print()
print("Repartition des labels (train) :")
print(features.label_distribution(train_df))
print("Repartition des labels (test) :")
print(features.label_distribution(test_df))


## 5. Baseline naive (classe majoritaire)

Predit systematiquement la classe majoritaire du train (`positif`). Sert
de reference plancher : tout modele doit faire mieux que cela pour
justifier sa complexite.


In [ ]:
baseline_metrics = model_module.evaluate_baseline_majority(train_df, test_df)
print(baseline_metrics)


## 6. Baseline classique (TF-IDF + regression logistique)

TF-IDF (uni + bi-grammes, `class_weight="balanced"` pour compenser le
desequilibre des classes) + regression logistique. Rapide a entrainer,
sert de reference "classique" avant d'investir dans le fine-tuning d'un
transformer.


In [ ]:
tfidf_pipeline, tfidf_metrics, tfidf_y_pred = model_module.train_tfidf_logreg(
    train_df, test_df
)
print(tfidf_metrics)
print()
print(
    features.compute_classification_report(test_df["label"].to_numpy(), tfidf_y_pred)
)


## 7. Fine-tuning DistilBERT -- smoke test (~100 exemples)

Conformement a `CLAUDE.md` ("tester sur ~100 exemples avant
l'entrainement complet"), on valide d'abord la plomberie complete
(tokenisation, forme des tenseurs, API Trainer, ponderation des classes)
sur un tres petit sous-echantillon et une seule epoque, avant d'investir
le temps CPU necessaire a l'entrainement complet (section 8). Les
metriques ci-dessous ne sont **pas representatives** (echantillon minuscule,
1 seule epoque) : seul le bon fonctionnement du pipeline compte ici.


In [ ]:
small_train = train_df.sample(n=100, random_state=RANDOM_STATE)
small_test = test_df.sample(n=50, random_state=RANDOM_STATE)

_, _, smoke_metrics, _ = model_module.train_distilbert(
    small_train,
    small_test,
    output_dir=model_module.MODELS_DIR / "_smoke_test_checkpoints",
    num_train_epochs=1.0,
    per_device_batch_size=8,
)
print("Smoke test (non representatif) :", smoke_metrics)
print("Plomberie validee.")


## 8. Fine-tuning DistilBERT -- entrainement complet

**Choix assume** : le fine-tuning complet (2 epoques, batch=16, sur les
~1 665 exemples d'entrainement) a ete execute au prealable en arriere-plan
via `python -m nlp.training.model` (memes fonctions que ci-dessus,
`random_state=42`), pour un cout mesure d'environ 80-90 minutes CPU (pas
de GPU disponible sur cette machine). Ce notebook **charge le modele deja
entraine** plutot que de relancer un fine-tuning complet en ligne, afin de
rester executable en quelques minutes -- conformement a l'esprit de la
consigne de prioriser la rigueur (split, analyse d'erreurs) plutot que le
temps de calcul consacre au tuning/entrainement. Le modele charge est
strictement celui qui sera exporte comme modele de production (meme
artefact, memes poids).


In [ ]:
import glob

candidate_dirs = sorted(glob.glob(str(model_module.MODELS_DIR / "distilbert_sentiment_v*")))
assert candidate_dirs, (
    "Aucun modele distilbert sauvegarde trouve -- "
    "executez d'abord `python -m nlp.training.model`."
)
model_dir = candidate_dirs[-1]
print(f"Modele charge : {model_dir}")

bundle = model_module.load_model(model_dir)
print(f"Metriques enregistrees a l'entrainement : {bundle.metrics}")

distilbert_metrics, distilbert_y_pred = model_module.evaluate_saved_distilbert(
    bundle, test_df
)
print(f"Metriques recalculees sur le test set (ce notebook) : {distilbert_metrics}")


## 9. Comparaison des modeles


In [ ]:
comparison = pd.DataFrame(
    [
        {"modele": "baseline_majoritaire", **baseline_metrics},
        {"modele": "tfidf_logreg", **tfidf_metrics},
        {"modele": "distilbert", **distilbert_metrics},
    ]
)
baseline_f1 = baseline_metrics["macro_f1"]
comparison["amelioration_vs_baseline_naive_%"] = (
    (comparison["macro_f1"] - baseline_f1) / baseline_f1 * 100
).round(2)
print(comparison.to_string(index=False))

f1_ok = distilbert_metrics["macro_f1"] > model_module.F1_MACRO_THRESHOLD
acc_ok = distilbert_metrics["accuracy"] > model_module.ACCURACY_THRESHOLD
print()
print(f"Seuil F1 macro > {model_module.F1_MACRO_THRESHOLD} : {f1_ok}")
print(f"Seuil accuracy > {model_module.ACCURACY_THRESHOLD} : {acc_ok}")


## 10. Matrice de confusion (modele retenu)


In [ ]:
y_true = test_df["label"].to_numpy()
cm_df = features.compute_confusion_matrix(y_true, distilbert_y_pred)
print(cm_df)

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_true, distilbert_y_pred, labels=features.LABELS, ax=ax, colorbar=False
)
plt.title("Matrice de confusion -- DistilBERT (test set)")
plt.tight_layout()
plt.show()


## 11. F1 par classe


In [ ]:
report_df = features.compute_classification_report(y_true, distilbert_y_pred)
print(report_df)

per_class = report_df.loc[features.LABELS, "f1-score"]
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(per_class.index, per_class.values, color=["indianred", "gray", "seagreen"])
ax.axhline(model_module.F1_MACRO_THRESHOLD, color="black", linestyle="--", linewidth=1)
ax.set_title("F1 par classe -- DistilBERT")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## 12. Exemples mal classes (commentes)


In [ ]:
misclassified = features.extract_misclassified(
    test_df.reset_index(drop=True), y_true, distilbert_y_pred, max_chars=280
)
print(f"{len(misclassified)} exemples mal classes sur {len(test_df)} ({100*len(misclassified)/len(test_df):.1f}%)")
misclassified.head(10)


## 13. Discussion des limites

*(section completee ci-dessous avec les chiffres reels apres execution)*


## 14. Conclusion

*(section completee ci-dessous avec les chiffres reels apres execution)*
